In [ ]:
import tensorflow as tf
import pandas as pd
from tensorflow import keras
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt

print(tf.__version__)

In [ ]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/sms/train-data.tsv
!wget https://cdn.freecodecamp.org/project-data/sms/valid-data.tsv

train_file_path = "train-data.tsv"
test_file_path = "valid-data.tsv"

In [ ]:
# Convert labels to numbers
train_df = pd.read_csv(train_file_path, sep='\t', names=['type', 'text'])
test_df = pd.read_csv(test_file_path, sep='\t', names=['type', 'text'])

train_df['type'] = train_df['type'].map({'ham': 0, 'spam': 1})
test_df['type'] = test_df['type'].map({'ham': 0, 'spam': 1})

train_labels = train_df.pop('type')
test_labels = test_df.pop('type')

In [ ]:
from tensorflow.keras.layers import TextVectorization

VOCAB_SIZE = 1000
MAX_LEN = 100

vectorizer = TextVectorization(
    max_tokens=VOCAB_SIZE,
    output_mode='int',
    output_sequence_length=MAX_LEN
)

# Adapt the vectorizer to the training text
vectorizer.adapt(train_df['text'].values)

In [ ]:
model = tf.keras.Sequential([
    vectorizer,
    tf.keras.layers.Embedding(VOCAB_SIZE, 64, mask_zero=True),
    tf.keras.layers.LSTM(64),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid') # Sigmoid for binary 0-1 output
])

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

# Convert the text series to numpy arrays to avoid the 'object' dtype error
history = model.fit(
    x=train_df['text'].values,
    y=train_labels.values,
    epochs=10,
    validation_data=(test_df['text'].values, test_labels.values)
)

In [ ]:
def predict_message(pred_text):
    input_tensor = tf.constant([pred_text], dtype=tf.string)

    # Run the prediction
    prediction_prob = model.predict(input_tensor, verbose=0)[0][0]

    # 0-0.5 is ham, 0.5-1.0 is spam
    label = "spam" if prediction_prob >= 0.5 else "ham"

    return [float(prediction_prob), label]

# Manual test
pred_text = "how are you doing today?"
prediction = predict_message(pred_text)
print(prediction)

In [ ]:
# Run this cell to test your function and model. Do not modify contents.
def test_predictions():
  test_messages = ["how are you doing today",
                   "sale today! to stop texts call 98912460324",
                   "i dont want to go. can we try it a different day? available sat",
                   "our new mobile video service is live. just install on your phone to start watching.",
                   "you have won £1000 cash! call to claim your prize.",
                   "i'll bring it tomorrow. don't forget the milk.",
                   "wow, is your arm alright. that happened to me one time too"
                  ]

  test_answers = ["ham", "spam", "ham", "spam", "spam", "ham", "ham"]
  passed = True

  for msg, ans in zip(test_messages, test_answers):
    prediction = predict_message(msg)
    if prediction[1] != ans:
      passed = False

  if passed:
    print("You passed the challenge. Great job!")
  else:
    print("You haven't passed yet. Keep trying.")

test_predictions()
